# 05 — Addressing CNN Overfitting

**Model Lead follow-up.** The published CNN baseline (`cnn_baseline.keras`, test accuracy 88.06%) showed a clear overfitting pattern: training accuracy climbed to ~96% while validation plateaued at ~87–89%, with validation loss flattening after epoch ~6–9 depending on batch size. This notebook tests whether **data augmentation + early stopping** reduces that gap, compared against the original baseline.

| Section | What happens |
|---|---|
| 0 | Setup |
| 1 | Rebuild dataset |
| 2 | Train the augmented model (early stopping, random flips/rotation/zoom/contrast) |
| 3 | Compare against the published baseline |
| 4 | Push results to Hugging Face |

## 0. Setup

In [ ]:
!git clone https://github.com/neuroarcane/dental-cavity-detector.git
%cd dental-cavity-detector
!pip install -r requirements.txt
!nvidia-smi

In [ ]:
from pathlib import Path

data_root = Path('/content/dental_data')
data_root.mkdir(parents=True, exist_ok=True)
!cp -r "data/raw/Dental X-ray.v1i.yolov11" {data_root}/
!cp -r "data/raw/Dental X-Ray Panoramic Dataset" {data_root}/
(data_root / "Dental X-ray.v1i.yolov11.zip.extracted").touch()
(data_root / "Dental X-Ray Panoramic Dataset.zip.extracted").touch()

from src.data.prepare_dataset import merge_datasets, print_merge_summary
from src.data.split import remove_exact_duplicates, stratified_resplit
from src.data.balance import oversample_minority_classes

merge_datasets()
remove_exact_duplicates()
stratified_resplit(train_frac=0.7, valid_frac=0.15, test_frac=0.15)
oversample_minority_classes(split='train', target_ratio=0.5, max_duplicates_per_image=5)

!cat data/processed/data.yaml

## 1. Rebuild the dataset pipeline

In [ ]:
import yaml, json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR = Path('data/processed')
IMG_SIZE = 128
SEED = 42

with open(DATA_DIR / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)
yaml_class_names = data_yaml['names']

CLASS_NAMES = ['Cavity', 'Filling', 'Crown', 'Impacted Tooth']
PRIORITY = ['Cavity', 'Crown', 'Impacted Tooth', 'Filling']
NUM_CLASSES = len(CLASS_NAMES)

def build_singlelabel_index(split_dir):
    images_dir, labels_dir = split_dir / 'images', split_dir / 'labels'
    rows = []
    for img_path in sorted(images_dir.glob('*.*')):
        label_path = labels_dir / f'{img_path.stem}.txt'
        present = set()
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    present.add(yaml_class_names[int(line.split()[0])])
        chosen = PRIORITY[-1]
        for cls in PRIORITY:
            if cls in present:
                chosen = cls
                break
        rows.append({'path': str(img_path), 'class_id': CLASS_NAMES.index(chosen)})
    return pd.DataFrame(rows)

train_df = build_singlelabel_index(DATA_DIR / 'train')
val_df   = build_singlelabel_index(DATA_DIR / 'valid')
test_df  = build_singlelabel_index(DATA_DIR / 'test')
print('train:', len(train_df), ' val:', len(val_df), ' test:', len(test_df))

def make_dataset(df, batch_size=32, shuffle=False):
    paths, labels = df['path'].values, df['class_id'].values
    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache()
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

present_classes = np.unique(train_df['class_id'].values)
weights = compute_class_weight(class_weight='balanced', classes=present_classes, y=train_df['class_id'].values)
class_weight = {i: 1.0 for i in range(NUM_CLASSES)}
class_weight.update(dict(zip(present_classes, weights)))
print('Class weights:', {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

In [ ]:
from huggingface_hub import hf_hub_download, HfApi, login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
HF_REPO_ID = 'aparnamohankumar/dental-cavity-detector'
api = HfApi()

## 2. Train with augmentation + early stopping

Same architecture as the baseline, with a data augmentation layer added at the input, and training capped by early stopping (on validation loss) instead of a fixed epoch count.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def build_cnn_baseline_augmented(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    return models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])

train_ds = make_dataset(train_df, batch_size=32, shuffle=True)
val_ds   = make_dataset(val_df, batch_size=32)
test_ds  = make_dataset(test_df, batch_size=32)

model_aug = build_cnn_baseline_augmented()
model_aug.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history_aug = model_aug.fit(
    train_ds, validation_data=val_ds, epochs=30,
    class_weight=class_weight, callbacks=[early_stop]
)

In [ ]:
test_loss_aug, test_acc_aug = model_aug.evaluate(test_ds, verbose=0)
print(f'Augmented model — test_acc: {test_acc_aug:.4f}, test_loss: {test_loss_aug:.4f}')

train_val_acc_gap_aug = history_aug.history['accuracy'][-1] - history_aug.history['val_accuracy'][-1]
train_val_loss_gap_aug = history_aug.history['val_loss'][-1] - history_aug.history['loss'][-1]
print(f'Train-val accuracy gap: {train_val_acc_gap_aug:.3f}')
print(f'Train-val loss gap: {train_val_loss_gap_aug:.3f}')
print(f'Epochs actually run (early stopping): {len(history_aug.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_aug.history['loss'], label='train')
axes[0].plot(history_aug.history['val_loss'], label='val')
axes[0].set_title('Loss — Augmented + Early Stopping'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(history_aug.history['accuracy'], label='train')
axes[1].plot(history_aug.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy — Augmented + Early Stopping'); axes[1].set_xlabel('epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_augmented_curve.png', dpi=150)
plt.show()

## 3. Compare against the published baseline

Baseline (no augmentation, fixed 15 epochs, batch 32): test accuracy 88.06%, test loss 0.9863, train-val accuracy gap ~0.09, train-val loss gap ~0.55–0.6 (see `03_cnn_baseline.ipynb`).

In [ ]:
baseline = {'test_accuracy': 0.8806, 'test_loss': 0.9863}

comparison = pd.DataFrame([
    {'model': 'Baseline (no augmentation)', 'test_accuracy': baseline['test_accuracy'], 'test_loss': baseline['test_loss']},
    {'model': 'Augmented + early stopping', 'test_accuracy': test_acc_aug, 'test_loss': test_loss_aug},
])
comparison

## 4. Push results to Hugging Face

In [ ]:
comparison.to_csv('cnn_augmentation_comparison.csv', index=False)
model_aug.save('cnn_augmented.keras')

api.upload_file(path_or_fileobj='cnn_augmentation_comparison.csv', path_in_repo='cnn_augmentation_comparison.csv', repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj='cnn_augmented_curve.png', path_in_repo='cnn_augmented_curve.png', repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj='cnn_augmented.keras', path_in_repo='cnn_augmented.keras', repo_id=HF_REPO_ID)
print('Pushed.')

## Notes for the report

- If the train-val gap shrank meaningfully compared to baseline (~0.09 accuracy, ~0.55–0.6 loss), augmentation + early stopping is a real fix worth adopting going forward.
- If the gap is similar, that's still a useful finding: it suggests the overfitting is more structural (model capacity, or genuine limits of the dataset size) than something augmentation alone can fix, and would point toward reducing model capacity or adding stronger regularization (e.g. higher dropout, L2 weight decay) as the next thing to try.